# Day 3: Tissue organisation, communication, interpretation

This session is **self-contained** and runs on the small 6 GB machine. We use a pre-annotated single-cell spatial dataset (seqFISH mouse embryo). Nothing here depends on Days 1 to 2.

**You will learn to**
- read tissue organisation with neighbourhood enrichment and co-occurrence;
- define niches from neighbourhood composition;
- score ligand-receptor interactions;
- rank spatially variable genes and interpret a finding end to end.

### Setup (run first)

On the course servers this does nothing, since the packages are pre-installed. On Google Colab or a fresh laptop it installs what is missing. Data downloads automatically the first time it is needed, except the Xenium bundle on Day 2 (see the README).

In [ ]:
import importlib, subprocess, sys
_req = [('scanpy','scanpy'), ('squidpy','squidpy'), ('leidenalg','leidenalg'),
        ('igraph','igraph'), ('pyclustree','pyclustree'),
        ('scikit-image','skimage'), ('imageio','imageio')]
_missing = [pip for pip, mod in _req if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing], check=False)
print('environment ready')

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "axes.titlesize": 11,
    "font.weight": "normal",
    "figure.dpi": 110,
})

def _data_dir():
    env = os.environ.get("SPATIAL_COURSE_DATA")
    if env:
        return Path(env)
    shared = Path("/home/shared/spatial_course_data")
    if shared.exists():
        return shared
    local = Path.home() / "spatial_course_data"
    local.mkdir(parents=True, exist_ok=True)
    return local

DATA_DIR = _data_dir()
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

def load_visium():
    p = DATA_DIR / "visium_lymph_node.h5ad"
    if p.exists():
        adata = sc.read_h5ad(p)
    else:
        adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
        try:
            adata.write(p)
        except Exception:
            pass
    adata.var_names_make_unique()
    return adata

def load_seqfish():
    p = DATA_DIR / "seqfish_embryo.h5ad"
    if p.exists():
        return sc.read_h5ad(p)
    adata = sq.datasets.seqfish()
    try:
        adata.write(p)
    except Exception:
        pass
    return adata

def xenium_dir():
    d = DATA_DIR / "xenium_kidney"
    if (d / "cell_feature_matrix.h5").exists():
        return d
    raise FileNotFoundError(
        f"Xenium bundle not found in {d}. See the README (Data): download the 10x "
        "Xenium Output Bundle once and unzip it there, or run scripts/prefetch_data.py."
    )

print("data dir:", DATA_DIR)

## 1. Load the annotated dataset

Cell types are already provided, so we go straight to organisation.

In [ ]:
adata = load_seqfish()
ct_col = 'celltype_mapped_refined'
if ct_col not in adata.obs:
    ct_col = next(c for c in adata.obs.columns if 'celltype' in c.lower() or 'cell_type' in c.lower())
print('cell type column:', ct_col)
adata

In [ ]:
sq.pl.spatial_scatter(adata, color=ct_col, shape=None, size=2)

### A closer look (spatial visualization)

Before any statistics, look at the section a few ways: spotlight a couple of cell types with `groups=` to see how they are laid out, and crop to a coordinate box to read local architecture. The same categorical-vs-continuous and `size`/`alpha` habits from Days 1-2 apply here.

In [ ]:
top_types = adata.obs[ct_col].value_counts().index[:3].tolist()
sq.pl.spatial_scatter(adata, color=ct_col, groups=top_types, shape=None, size=2)

In [ ]:
xy = adata.obsm['spatial']
sel = ((xy[:, 0] > np.quantile(xy[:, 0], 0.4)) & (xy[:, 0] < np.quantile(xy[:, 0], 0.6)) &
       (xy[:, 1] > np.quantile(xy[:, 1], 0.4)) & (xy[:, 1] < np.quantile(xy[:, 1], 0.6)))
sub = adata[sel].copy()
print(sub.n_obs, 'cells in crop')
sq.pl.spatial_scatter(sub, color=ct_col, shape=None, size=10)

## 2. Neighbourhood enrichment

> **Method note: what the score is, and its caveats.** For each pair of cell types, count adjacent cells on the graph, then permute labels to get a null and report $z = (O-\mu_\pi)/\sigma_\pi$. Permutation preserves composition but destroys spatial arrangement, so $z>0$ means the pair is adjacent more than chance (co-location) and $z<0$ means avoidance. The score is relative to the chosen graph and the section's composition, so a difference between samples can be a composition difference. Dedicated niche tools (CellCharter, BANKSY) turn the same idea into reusable domain labels; we start from the raw enrichment.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)
sq.gr.nhood_enrichment(adata, cluster_key=ct_col)
sq.pl.nhood_enrichment(adata, cluster_key=ct_col, figsize=(7, 7))

> **Reading the enrichment map.** Rows and columns are cell types, coloured by z-score. The diagonal is self-aggregation (does a type cluster together). Off the diagonal, warm means two types are neighbours more than chance (co-location), cool means avoidance. The magnitude is a permutation signal-to-noise, not a probability, and it depends on the graph and composition, so confirm a striking pair by plotting both types in space.

> **Exercise 1.** From the matrix, name two cell-type pairs that co-locate and two that avoid each other. For one co-locating pair, give a one-line biological guess.

In [ ]:
# your code here


## 3. Co-occurrence across distance

The co-occurrence score for types $A,B$ at distance $d$ is $\dfrac{p(B \mid A, d)}{p(B)}$: above 1 means $B$ is enriched within distance $d$ of $A$. Plotting it against $d$ shows whether an association is short-range contact or broad regional co-occurrence.

In [ ]:
sq.gr.co_occurrence(adata, cluster_key=ct_col)
focus = adata.obs[ct_col].value_counts().index[0]
sq.pl.co_occurrence(adata, cluster_key=ct_col, clusters=focus, figsize=(7, 4))
print('focus:', focus)

> **Reading co-occurrence.** The curve is $p(B \mid A, d)/p(B)$ against distance $d$: above 1 means $B$ is enriched around $A$ at that range, 1 is no association, below 1 is depletion. A short-range peak is contact; a long plateau above 1 is regional co-occurrence; a decaying curve is a gradient away from $A$.

## 4. Niches from neighbourhood composition

> **Method note: the recipe, the maths, and the assumptions.** For each cell $i$, form the cell-type composition of its spatial neighbourhood, $h_i \in \mathbb{R}^T$ with $\sum_t h_{it}=1$ over the $T$ types, then cluster the $h_i$ with k-means, which minimises $\sum_k \sum_{i \in C_k} \lVert h_i - \mu_k \rVert^2$. Each cluster is a recurring multicellular niche. Assumptions: the neighbourhood composition captures the niche, and k-means prefers roughly round, equal-spread clusters in composition space, so elongated niches can be split. There is no true $k$: scan a few values and judge by interpretability (a silhouette score or the composition crosstab can guide you). Learned methods (BANKSY, CellCharter, SpaGCN, STAGATE) can capture subtler domains but add assumptions and compute. This is the *composition*-clustering route to multicellular regions; Day 2 §9 shows the complementary *feature-augmentation* route (stacking neighbour-averaged expression before clustering), two views of the same goal.

In [ ]:
from sklearn.cluster import KMeans
def niches(adata, ct_col, k, seed=0):
    conn = adata.obsp['spatial_connectivities']
    onehot = pd.get_dummies(adata.obs[ct_col])
    comp = conn.dot(onehot.to_numpy(dtype=float))
    denom = comp.sum(1, keepdims=True); denom[denom == 0] = 1.0
    comp = comp / denom
    labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(comp)
    return pd.Categorical(labels.astype(str))
adata.obs['niche'] = niches(adata, ct_col, k=6)
sq.pl.spatial_scatter(adata, color='niche', shape=None, size=2)

In [ ]:
pd.crosstab(adata.obs['niche'], adata.obs[ct_col], normalize='index').round(2)

> **Exercise 2.** Recompute niches with k=4 and k=8 (use the `niches` helper). Compare the crosstabs. Which k gives interpretable, distinct compositions, and why did you pick it?

In [ ]:
# your code here


## 5. Cell-cell communication

> **Method note: how scoring works, and what to trust.** For a ligand $L$ and receptor $R$ and cell-type groups $A,B$, the CellPhoneDB-style score is the product (or mean) of mean expression, $s = \bar{L}_A \cdot \bar{R}_B$, and significance comes from permuting the cell-type labels many times to build a null for $s$. squidpy `ligrec`, LIANA, and CellPhoneDB share this logic; NicheNet adds downstream target genes; spatial-aware methods (COMMOT, NCEM) add distance or neighbour constraints. Caveats: expression is a proxy for activity, a small panel covers few pairs, the p-value depends on the permutation scheme, and segmentation errors can move a ligand to the wrong cell. Read results as hypotheses. We run offline with a small shipped table; drop `interactions` to use a full database online.

In [ ]:
lr = pd.read_csv(Path('data/ligand_receptor_pairs.csv'))
present = lr[lr['source'].isin(adata.var_names) & lr['target'].isin(adata.var_names)]
print(f'{len(present)} of {len(lr)} pairs in panel')
present.head()

In [ ]:
if len(present):
    try:
        sq.gr.ligrec(adata, cluster_key=ct_col,
                     interactions=present[['source', 'target']],
                     use_raw=False, n_perms=100, seed=0)
        print('ligrec done; results in adata.uns')
    except Exception as e:
        print('ligrec skipped:', e)
else:
    print('No shipped pairs intersect this panel; add your own or run online.')

> **Exercise 3.** Pick a source and target cell type and inspect the ligrec result for that pair (means and p-values). Name one interaction that looks plausible and say what neighbour relationship would support it.

In [ ]:
# your code here


## 6. Spatially variable genes

In [ ]:
sq.gr.spatial_autocorr(adata, mode='moran')
top = adata.uns['moranI'].head(4).index.tolist()
sq.pl.spatial_scatter(adata, color=top, shape=None, size=2)

> **Capstone.** Pick one finding from today (an enriched pair, a niche, a top spatial gene). Read it end to end: what does the statistic claim in plain words; is it consistent across the section or driven by one region; what could confound it (composition, segmentation, panel limits); and what analysis or experiment would test it next? Write a short paragraph and be ready to defend it.

In [ ]:
# your code here


### Where to go next

The `scverse` stack (`scanpy`, `anndata`, `squidpy`, `spatialdata`) scales the same workflow from one section to a cohort. Beyond this course: multi-sample integration and reference mapping, deconvolution for spot data, and the learned niche and communication methods named in the notes.

Thank you.